In [157]:
import pandas as pd
from pathlib import Path
import os
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re # regular expressions

In [158]:
# Import cvs files nd place into dictinoary

print(os.getcwd())
print(os.listdir("."))

# place all data sources into folder call "data" alongside this .ipynb file
data_location = Path("data")


# Create a dictionary into which we wil aec the four dataframes and align them with the file names, for ease of access
# This will save us time cleaning the dfs individually with any repitive tasks 
data_sources= {}

print(data_location)

for i in sorted(data_location.glob("*.csv")):
    #strip back file name to just prefix, split at every dot and take first item and convert to lower case for consistency
    stripped_dfname= i.stem.split(".")[0].lower()
    # read csv file into a dataframe - red as utf8 and strip the hidden byte orser mark... as gettng corrupt characters..
    df = pd.read_csv(i, header=0, encoding="utf-8-sig")
    # standardise column names
    df.columns =df.columns.str.lower().str.replace(" ", "_")
    # stpre thb dataframe in the dictionary
    data_sources[stripped_dfname] = df
    # have a llok to make sure file names match into dataframe name in the dictionary and check shape to verify df is good
    print(f"{stripped_dfname} : {df.shape} <-{i.name}")


# Call the dictionary key names to show the dataframes present..
data_sources.keys()




e:\GG\Msc\SPEC9995_data_visualisation_marisa llorens-salvador_tues\github\spec9995_data_vis_marissa_tues\assignment 1\working
['working_dea_1.ipynb', 'steps_log.txt', 'data', 'data_overview.txt', 'eda_core_v1.png', 'eda_net_debt_waterfall_v1.png', 'core_features.csv', 'eda_core_v2.png', 'eda_core_v3.png', 'working_dea_v2.ipynb', 'tableau_all_long.csv', 'tableau_core_long.csv', 'tableau_core_wide.csv', 'working_v3.ipynb', '~WRL1103.tmp', 'assignment1_cso_hvd_eamonn_kelly_D24127620.pdf.docx', 'exported', 'images', 'assignment1_cso_hvd_eamonn_kelly_D24127620)_v2.pdf.docx', '~$signment1_cso_hvd_eamonn_kelly_D24127620)_v3.pdf.docx', 'CSO -  Revenue vs Expenditures.twb', 'CSO -  Surplus vs Deficit.twb', 'CSO -  Gross and Net debt with Cash & Bond Assets.twb', 'CSO -  Tax Composition - Corp_Tax_Income_Tax_VAT.twb', '~CSO -  Tax Composition - Corp_Tax_Income_Tax_VAT__31168.twbr', '~CSO -  Gross and Net debt with Cash & Bond Assets__5616.twbr', 'assignment1_cso_hvd_eamonn_kelly_D24127620)_v3.pd

dict_keys(['gfa01', 'gfa02', 'gfa03', 'gfa04', 'gfq02'])

In [159]:
# Creating a dictionary definition of what to keep from the downloaded csv files, mapping the retained data to new feature names in dictionary key value pairs
keep = {
    "gfa01": {
        "General Government transactions - Revenue - ESA2010 Code (TR)": "total_revenue",
        "General Government transactions - Expenditure - ESA2010 Code (TE)": "total_expenditure",
        "General Government Surplus/Deficit - ESA2010 Code (B9)": "surplus_deficit",
        "General Government transactions - Revenue - Capital transfer revenue (excluding taxes) - ESA2010 Code (D9N)": "one_off_capital_receipt",
        "General Government transactions - Expenditure - Interest (excluding FISIM) - ESA2010 Code (D41)": "interest_debt",
    },
    "gfa02": {
        "Financial net worth at market value, excluding pension liabilities - EDP debt instruments (Asset) - ESA2010 Code (F2 + F3 + F4)": "cash_bond_assets",
        "Gross General Government Debt (EDP face value) - Gross General Government Debt - ESA2010 Code (GGDebt)": "gross_debt",
        "Gross General Government Debt (EDP face value) - Net General Government Debt - ESA2010 Code (GGNetDebt)": "net_debt",
    },
    "gfa03": {
        "Revenue - Total Taxes - ESA2010 Code (D2 + D5 + D91)": "total_tax",
        "Revenue - Corporation tax - ESA2010 Code (D51 part of)": "ct",
        "Revenue - Income tax - ESA2010 Code (D51 part of)": "income_tax",
        "Revenue - Taxes on products of which Value-Added Tax - ESA2010 Code (D211)": "vat",
    },
    "gfa04": {
        "Expenditure - Gross fixed capital formation - ESA2010 Code (P51)": "public_investment",
        "Expenditure - Total Social benefits - ESA2010 Code (D62 + D632)": "social_benefits",
        "Expenditure - Other capital transfers - ESA2010 Code (D99)": "capital_transfers_paid",
    },
}

In [160]:
# filter and rename the features fromt he dictioary that we want to keep

# create a list to store the selected data series, each table resutls get stacked in this, at the end of each other, sequentially
stacked_gfadata_list = []
for table, mapping in keep.items():
    # we'll sue a working copy dictionary ut 
    df = data_sources[table].copy()
    # use regex to strip any double spaces, >=1 space replaced with 1 space, and remove leding an trailing
    df["item"] = df["item"].str.replace(r"\s+", " ", regex=True).str.strip()  
    # deal with any missing values in our mapping, using sets to remove items nto in the keep dict we defined earlier
    missing = set(mapping) - set(df["item"])       
    if missing:
        print(f"WARNING {table}: not found -> {missing}")

    # keep only the rows in that mapping
    df = df[df["item"].isin(mapping)].copy()
    # map each rows label and gets short name from dict item thats put in df series
    df["feature_name"] = df["item"].map(mapping)
    # create new label names for the df and put in the values
    df["orig_org_source"], df["orig_file_source"], df["is_forecast"] = "CSO", table.upper(), False
    # append to the defined list
    stacked_gfadata_list.append(df)
    # pritn stament - troubleshooting...check what we have . print the mapping
    # table is the name in the dictionary for the csv filename abbrev used in the dictionary
    print(f"{table} : {df.shape[0]} rows <- {len(mapping)} features over a period of {df['year'].nunique()} years "
          f"({df['year'].min()} to {df['year'].max()})")

# put it all table into new dataframe called df_cso
df_cso = pd.concat(stacked_gfadata_list, ignore_index=True)[
    ["year", "feature_name", "value", "unit", "orig_org_source", "orig_file_source", "is_forecast"]]

# print the totals - the shape of the dataframe
print("\ndf_cso:", df_cso.shape, "| total number of features", df_cso["feature_name"].nunique())
display(df_cso.head())

gfa01 : 155 rows <- 5 features over a period of 31 years (1995 to 2025)
gfa02 : 78 rows <- 3 features over a period of 26 years (2000 to 2025)
gfa03 : 104 rows <- 4 features over a period of 26 years (2000 to 2025)
gfa04 : 78 rows <- 3 features over a period of 26 years (2000 to 2025)

df_cso: (415, 7) | total number of features 15


,year,feature_name,value,unit,orig_org_source,orig_file_source,is_forecast
0,1995,total_revenue,20959.0,Euro Million,CSO,GFA01,False
1,1995,one_off_capital_receipt,911.0,Euro Million,CSO,GFA01,False
2,1995,total_expenditure,22096.0,Euro Million,CSO,GFA01,False
3,1995,interest_debt,2814.0,Euro Million,CSO,GFA01,False
4,1995,surplus_deficit,-1137.0,Euro Million,CSO,GFA01,False


In [161]:
print(df_cso.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415 entries, 0 to 414
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              415 non-null    int64  
 1   feature_name      415 non-null    object 
 2   value             415 non-null    float64
 3   unit              415 non-null    object 
 4   orig_org_source   415 non-null    object 
 5   orig_file_source  415 non-null    object 
 6   is_forecast       415 non-null    bool   
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 20.0+ KB
None


In [162]:
# EXploratory Data Analysis

print(df_cso.describe())

              year          value
count   415.000000     415.000000
mean   2011.566265   37157.248193
std       8.169687   52265.294915
min    1995.000000  -53764.000000
25%    2005.000000    3887.000000
50%    2012.000000   15514.000000
75%    2019.000000   44555.000000
max    2025.000000  235182.000000


In [163]:
# quick look at data to see if its clean, no duplicates, null values unusual characters etc
# convert it to d first so can easily run some checks over it
# df=pd.DataFrame(processed_per_file_data)
# view various dataframe details
# checking for null values and specific unknown values
print ("Rows     : " , df_cso.shape[0])
print ("Columns  : " , df_cso.shape[1])
print('------')
print ("\nFeatures : \n" , df_cso.columns.tolist())
print('------')
print ("\nMissing values :  ", df_cso.isnull().sum().values.sum())
print('------')
print ("\nMissing values per column:  ", df_cso.isnull().sum())
print('------')
print('\nnum of dups in df   =   {}'.format(df_cso.duplicated().sum()))
print('------')
print ("\nUnique values :  \n", df_cso.nunique())
print('------')
print('\n null values = ', df_cso.isnull().values.any())
print('------')
df_cso_missing = (df_cso=='?').sum()
print('\"  ?   \" values present =  \n',df_cso_missing)
print('------')
df_cso_unknown = (df_cso=='unknown').sum()
print('\"   unknown   \" values present =  \n',df_cso_unknown)
print('------')
df_cso_na = df_cso.isin(['N/A', 'N\\A', 'NA', 'n/a', 'n\\A', 'na', 'N_A']).sum()
print('    NA     values present = \n', df_cso_na)
print('------')
df_cso_none = df_cso.isin(['None', 'none']).sum()
print('    none    values present = \n', df_cso_none)
print('------')
# regex https://stackoverflow.com/questions/2124010/grep-regex-to-match-non-ascii-characters
# count of rows per column containing non-ascii characters
rogue_ascii_df_cso = df_cso.apply(lambda s: s.astype(str).str.contains(r"[^\x00-\x7F]", regex=True, na=False).sum())
print("    non ascii chars   values present:\n", rogue_ascii_df_cso)

Rows     :  415
Columns  :  7
------

Features : 
 ['year', 'feature_name', 'value', 'unit', 'orig_org_source', 'orig_file_source', 'is_forecast']
------

Missing values :   0
------

Missing values per column:   year                0
feature_name        0
value               0
unit                0
orig_org_source     0
orig_file_source    0
is_forecast         0
dtype: int64
------

num of dups in df   =   0
------

Unique values :  
 year                 31
feature_name         15
value               413
unit                  1
orig_org_source       1
orig_file_source      4
is_forecast           1
dtype: int64
------

 null values =  False
------
"  ?   " values present =  
 year                0
feature_name        0
value               0
unit                0
orig_org_source     0
orig_file_source    0
is_forecast         0
dtype: int64
------
"   unknown   " values present =  
 year                0
feature_name        0
value               0
unit                0
orig_org_sourc

In [164]:
# checkl the -71,556 value in values.... can ahve negative values
display(df_cso.loc[df_cso["value"].idxmin()])
display(df_cso[df_cso["value"] < 0].groupby("feature_name")["value"].agg(["min", "count"]))

year                           2010
feature_name        surplus_deficit
value                      -53764.0
unit                   Euro Million
orig_org_source                 CSO
orig_file_source              GFA01
is_forecast                   False
Name: 79, dtype: object

,min,count
feature_name,,
surplus_deficit,-53764.0,15


In [165]:
# craeting calculated features from CSO data
# NORM_START, NORM_END = 2000, 2014        # pre-windfall period for the "normal" CT share

# finction which will take 4 inputs i.e. df, new feature name, calcualted values for new feature (one per year), and unit value
def add_feature(df, name, series, unit):
    #Remove any existing rows for `name`, then append `series` (indexed by year) in df_cso format
    # keep all rows except  the new data row  "feature name" if it exists - allows to rerun, so if "feature aname " alreadyh exosts remove it
    df = df[df["feature_name"] != name]               
    # remove any blank values in the row  , some data i missing for 1995 to 1999, in some cases. tax data start in 2000
    s = series.dropna()
    # we'll defnie what goes into the existing columns where blanks will appear for the new feature
    rows = pd.DataFrame({
        "year": s.index,
        "feature_name": name,
        "value": s.values,
        "unit": unit,
        "orig_org_source": "CSO",
        "orig_file_source": "Calculated", # label it as claculated
        "is_forecast": False,
    })
    return pd.concat([df, rows], ignore_index=True)

In [166]:
# crate other tax value - subtract Corp Tax + Income tax + VAY tax from total tax to get toehr tax amounts...
# we pivot the dataframe by year to get year per row and the features per column, into a Wide view 
# # this wll make it easier and give us one column per feature, one row per year, for calculating across features
# we keep thsi separate from the dataframe we add to , we calcualet off this and then add results to dataframe
df_cso_wide = df_cso.pivot(index="year", columns="feature_name", values="value")


In [167]:
# Add "other_tax" calculated feature to dataframe
# ceate  working dataframe off the existing one, to allow for rollback whilr troubleshooting
df_cso_1 = df_cso
df_cso_1 = add_feature(df_cso_1, "other_tax", df_cso_wide["total_tax"] - df_cso_wide["ct"] - df_cso_wide["income_tax"] - df_cso_wide["vat"], "Euro Million")




In [168]:
df_cso_1.shape
# print the totals - the shape of the dataframe
print("\ndf_cso_1:", df_cso_1.shape, "| total number of features", df_cso_1["feature_name"].nunique())


df_cso_1: (441, 7) | total number of features 16


In [169]:
# Add Coroporation tax percentage of total tax value i.e. its share of total tax
# thi is a value we wil want to plot later on, so calculating and adding now

# creat working dataframe off existing one
df_cso_2 = df_cso_1

df_cso_2 = add_feature(df_cso_2, "ct_pct_tot_tax", df_cso_wide["ct"] / df_cso_wide["total_tax"] * 100, "Percent")

In [170]:
print("\ndf_cso_2:", df_cso_2.shape, "| total number of features", df_cso_2["feature_name"].nunique())


df_cso_2: (467, 7) | total number of features 17


In [ ]:
   # calculate the average value for 
   # NORM_START, NORM_END = 2000, 2014        # pre-windfall period for the "normal" CT share
   
   # df_cso_wide_2 = df_cso_2.pivot(index="year", columns="feature_name", values="value")
   # print("other_tax check:", (df_cso_wide_2["ct"] + df_cso_wide_2["income_tax"] + df_cso_wide_2["vat"] + df_cso_wide_2["other_tax"] - df_cso_wide_2["total_tax"]).abs().max())
   # print(df_cso_wide_2["ct_pct_tot_tax"].loc[[2000, 2014, 2025]].round(1))
   # print("normal CT share 2000-2014:", round(w2["ct_pct_tot_tax"].loc[2000:2014].mean(), 2), "%")

other_tax check: 0.0
year
2000    13.3
2014     9.9
2025    29.5
Name: ct_pct_tot_tax, dtype: float64
normal CT share 2000-2014: 11.86 %


In [173]:
# Create calculated feature "ct_diff_from_xpctd "  corporation tax value difference from expected CT value i.e. the excess CT
# we'll sue the wide version of the data frame crated earlier for ease of calculation then ppulate dataframe wiht the calculations
# to calculate the normal corporation tax we take a period of normal CT and get the average ver that persio
# thsi comes to approx 11.86% as outlined in previous cell

# creating workign wide and normal dataframe
df_cso_wide_3 =  df_cso_wide_2 
df_cso_3 = df_cso_2

# pre-windfall time basedon the oeriod 2000 to 2014
norm_start = 2000
norm_end = 2014 

norm = df_cso_wide_3["ct_pct_tot_tax"].loc[norm_start:norm_end].mean() /100

# have a lok at the valoues for 2000, 2014 and 2025
print(df_cso_wide_3["ct_pct_tot_tax"].loc[[2000, 2014, 2025]].round(1))
# get the average value over the 2000 to 2014 period
print("normal CT share 2000-2014:", round(df_cso_wide_3["ct_pct_tot_tax"].loc[2000:2014].mean(), 2), "%")


# creatign working datframes to alow rol back
# df_cso_wide_3 =  df_cso_wide_2 
# df_cso_3 = df_cso_2
# norm = (df_cso_wide["ct"] / df_cso_wide["total_tax"]).loc[NORM_START:NORM_END].mean()

ct_diff_from_xpctd = df_cso_wide_3["ct"] - norm * df_cso_wide_3["total_tax"]
df_cso_3 = add_feature(df_cso_2, "ct_diff_from_xpctd", ct_diff_from_xpctd, "Euro Million")

print(ct_diff_from_xpctd.loc[[2002, 2011, 2025]].round(0))



year
2000    13.3
2014     9.9
2025    29.5
Name: ct_pct_tot_tax, dtype: float64
normal CT share 2000-2014: 11.86 %
year
2002      922.0
2011    -1110.0
2025    19681.0
dtype: float64


In [174]:
print("\ndf_cso_3:", df_cso_3.shape, "| total number of features", df_cso_3["feature_name"].nunique())


df_cso_3: (493, 7) | total number of features 18


In [178]:
# create workign dfs
# need to sue calculation from previos one as se din this so making new df_cso_3 as working wide df to use for calculations
# df_cso_wide_4 =  df_cso_wide_3
df_cso_4 = df_cso_3

# craete budget surplus minus the excess corporation tax find value if over or under amount of revenu received
# norm = (df_cso_3["ct"] / df_cso_3["total_tax"]).loc[NORM_START:NORM_END].mean()

bdgt_srplus_minus_ct_diff_from_xpctd = df_cso_wide_4["surplus_deficit"] - ct_diff_from_xpctd.clip(lower=0) 
df_cso_4 = add_feature(df_cso_3, "bdgt_srplus_minus_ct_diff_from_xpctd", bdgt_srplus_minus_ct_diff_from_xpctd, "Euro Million")

print(bdgt_srplus_minus_ct_diff_from_xpctd.loc[[2011, 2022, 2025]].round(0))





year
2011   -23320.0
2022    -3462.0
2025    -9014.0
dtype: float64


In [179]:
print("\ndf_cso_4:", df_cso_4.shape, "| total number of features", df_cso_4["feature_name"].nunique())


df_cso_4: (519, 7) | total number of features 19


In [ ]:
print("\ndf_cso_3:", df_cso_3.shape, "| total number of features", df_cso_3["feature_name"].nunique())

In [52]:
# export to csv so cna load into Tableau and do explorattory data visualisations
# define the path and create the folder if it doesn' exist
export_location = Path("exported")
export_filename_cso = ("df_cso.csv")
export_location.mkdir(exist_ok=True)

df_cso.to_csv(export_location/export_filename_cso , index=False)
print(f" saved file : {(export_location/export_filename_cso).resolve()}")

 saved file : E:\GG\Msc\SPEC9995_data_visualisation_marisa llorens-salvador_tues\github\spec9995_data_vis_marissa_tues\assignment 1\working\exported\df_cso.csv
